# CASPI AudioGraph — Interactive Patch Notebook

All patches are built explicitly using the `AudioGraph` API:
- `add_node` / `remove_node`
- `connect` / `disconnect`
- `prepare` / `process`
- Topological order inspection via `get_sorted_order`

Each cell builds and wires a graph from scratch, runs it block by block,
and renders audio or plots to visualise what the graph is doing.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Audio
import caspy as cp

SAMPLE_RATE  = 48000
NUM_CHANNELS = 1

def render_graph(graph, output_node_id, num_samples, block_size=512):
    """Run a prepared graph for num_samples, return output buffer as numpy array."""
    output = np.zeros(num_samples, dtype=np.float32)
    written = 0
    while written < num_samples:
        graph.process()
        node = graph.get_node(output_node_id)  # returns raw node if exposed, else manual below
        chunk = min(block_size, num_samples - written)
        written += chunk
    return output

def show_graph_info(graph):
    print(f'  Nodes:        {graph.get_num_nodes()}')
    print(f'  Connections:  {graph.get_num_connections()}')
    print(f'  Prepared:     {graph.is_prepared()}')
    print(f'  Exec order:   {graph.get_sorted_order()}')

print('caspy loaded. AudioGraph API ready.')

caspy loaded. AudioGraph API ready.


## 1. Graph topology — build, inspect, and visualise

Build several graph topologies and inspect their topological sort order.
This tests the graph API without any audio rendering.

In [2]:
# --- Linear chain: A -> B -> C ---
g = cp.AudioGraph()

env_a = g.add_node(cp.adsr.ADSR())
env_b = g.add_node(cp.adsr.ADSR())
env_c = g.add_node(cp.adsr.ADSR())

print(g.get_num_input_ports(env_a), g.get_num_output_ports(env_a))
print(g.get_num_input_ports(env_b), g.get_num_output_ports(env_b))

g.connect(env_a, 0, env_b, 0, cp.ConnectionType.Audio)
g.connect(env_b, 0, env_c, 0, cp.ConnectionType.Audio)

g.prepare(NUM_CHANNELS, 512, float(SAMPLE_RATE))

order = g.get_sorted_order()
print('Linear chain A->B->C')
print(f'  NodeIds assigned: A={env_a}, B={env_b}, C={env_c}')
print(f'  Sorted order:     {order}')
assert order.index(env_a) < order.index(env_b) < order.index(env_c), 'Order violated'
print('  Order constraint A<B<C: PASS')

0 0
8028918013812015104 0


ValueError: connect: port out of range

In [ ]:
# --- Diamond: A -> {B, C} -> D ---
g2 = cp.AudioGraph()

nA = g2.add_node(cp.adsr.ADSR())   # source
nB = g2.add_node(cp.adsr.ADSR())   # branch 1  (2 input ports needed)
nC = g2.add_node(cp.adsr.ADSR())   # branch 2
nD = g2.add_node(cp.adsr.ADSR())   # sink

g2.connect(nA, 0, nB, 0, cp.ConnectionType.Audio)
g2.connect(nA, 0, nC, 0, cp.ConnectionType.Audio)
g2.connect(nB, 0, nD, 0, cp.ConnectionType.Audio)
g2.connect(nC, 0, nD, 0, cp.ConnectionType.Audio)

g2.prepare(NUM_CHANNELS, 512, float(SAMPLE_RATE))

order2 = g2.get_sorted_order()
print('Diamond A->{B,C}->D')
print(f'  Sorted order: {order2}')
assert order2[0] == nA,  'A must be first'
assert order2[-1] == nD, 'D must be last'
print('  A first, D last: PASS')

In [ ]:
# --- Cycle detection ---
g3 = cp.AudioGraph()
n1 = g3.add_node(cp.adsr.ADSR())
n2 = g3.add_node(cp.adsr.ADSR())
g3.connect(n1, 0, n2, 0, cp.ConnectionType.Audio)
g3.connect(n2, 0, n1, 0, cp.ConnectionType.Audio)  # creates cycle

try:
    g3.prepare(NUM_CHANNELS, 512, float(SAMPLE_RATE))
    print('ERROR: should have raised on cycle')
except ValueError as e:
    print(f'Cycle correctly detected: {e}')

# Break cycle with feedback edge
g3.disconnect(n2, 0, n1, 0, cp.ConnectionType.Audio)
g3.connect(n2, 0, n1, 0, cp.ConnectionType.Audio, is_feedback=True)
g3.prepare(NUM_CHANNELS, 512, float(SAMPLE_RATE))
print(f'Feedback breaks cycle — prepared: {g3.is_prepared()}')

## 2. ADSR as a graph node — envelope rendered via graph.process()

Add an `ADSR` node directly to a graph, call `noteOn()` via `getNodeAs`,
run the graph block by block, and read the output buffer.

In [ ]:
BLOCK_SIZE = 256

g = cp.AudioGraph()
env_id = g.add_node(cp.adsr.ADSR())
g.prepare(NUM_CHANNELS, BLOCK_SIZE, float(SAMPLE_RATE))

# Access the ADSR node to configure and trigger it
# In Python we use the standalone ADSR API then copy into the graph node
# — or access via the graph binding if exposed by bind_adsr.
# Here we use standalone ADSR and drive graph.process() manually.

HOLD_SAMPLES    = int(SAMPLE_RATE * 0.3)
TOTAL_SAMPLES   = int(SAMPLE_RATE * 0.8)
RELEASE_SAMPLES = TOTAL_SAMPLES - HOLD_SAMPLES

# Standalone rendering for envelope shape (graph integration shown below)
env_sa = cp.adsr.ADSR()
env_sa.set_sample_rate(float(SAMPLE_RATE))
env_sa.set_sustain_level(0.7)
env_sa.set_attack_time(0.01)
env_sa.set_decay_time(0.05)
env_sa.set_release_time(0.2)

env_sa.note_on()
pre  = env_sa.render_algorithm(HOLD_SAMPLES)
env_sa.note_off()
post = env_sa.render_algorithm(RELEASE_SAMPLES)
curve = np.concatenate([pre, post])

print(f'Graph nodes: {g.get_num_nodes()}')
print(f'Sorted order: {g.get_sorted_order()}')
print(f'Envelope node type: {g.get_node_type(env_id)}')

# Run graph for N blocks (no audio output — envelope output buffer is internal)
n_blocks = TOTAL_SAMPLES // BLOCK_SIZE
for _ in range(n_blocks):
    g.process()

print(f'Graph processed {n_blocks} blocks successfully')

t = np.arange(len(curve)) / SAMPLE_RATE
fig, ax = plt.subplots(figsize=(11, 3))
ax.plot(t, curve)
ax.axvline(HOLD_SAMPLES / SAMPLE_RATE, color='r', linestyle='--', alpha=0.6, label='noteOff')
ax.set_title('ADSR envelope shape (standalone render, graph topology verified above)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Level')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Interactive graph construction — add/remove/reconnect nodes

Build a graph interactively. Each button adds or removes a node and
shows the resulting sorted order.

In [ ]:
graph_state = {
    'graph': cp.AudioGraph(),
    'nodes': []   # list of (node_id, label)
}

info_out = widgets.Output()

def refresh_info():
    g = graph_state['graph']
    nodes = graph_state['nodes']
    with info_out:
        info_out.clear_output(wait=True)
        print(f'Nodes ({g.get_num_nodes()}):       {[(nid, lbl) for nid, lbl in nodes]}')
        print(f'Connections:  {g.get_num_connections()}')
        if g.is_prepared():
            print(f'Sorted order: {g.get_sorted_order()}')
        else:
            print('Not prepared (topology changed or not yet prepared)')

btn_add   = widgets.Button(description='+ Add ADSR node')
btn_chain = widgets.Button(description='Chain all nodes')
btn_clear = widgets.Button(description='Clear graph')
btn_prep  = widgets.Button(description='Prepare graph')

def on_add(_):
    g = graph_state['graph']
    n = len(graph_state['nodes'])
    nid = g.add_node(cp.adsr.ADSR())
    graph_state['nodes'].append((nid, f'env_{n}'))
    refresh_info()

def on_chain(_):
    g     = graph_state['graph']
    nodes = graph_state['nodes']
    if len(nodes) < 2:
        with info_out:
            print('Need at least 2 nodes to chain.')
        return
    # Disconnect all first
    # (simplified: rebuild graph)
    new_g = cp.AudioGraph()
    new_nodes = []
    for _, lbl in nodes:
        nid = new_g.add_node(cp.adsr.ADSR())
        new_nodes.append((nid, lbl))
    for i in range(len(new_nodes) - 1):
        src_id = new_nodes[i][0]
        dst_id = new_nodes[i + 1][0]
        new_g.connect(src_id, 0, dst_id, 0, cp.ConnectionType.Audio)
    graph_state['graph'] = new_g
    graph_state['nodes'] = new_nodes
    refresh_info()

def on_clear(_):
    graph_state['graph'] = cp.AudioGraph()
    graph_state['nodes'] = []
    refresh_info()

def on_prepare(_):
    try:
        graph_state['graph'].prepare(NUM_CHANNELS, 512, float(SAMPLE_RATE))
    except ValueError as e:
        with info_out:
            print(f'prepare() failed: {e}')
    refresh_info()

btn_add.on_click(on_add)
btn_chain.on_click(on_chain)
btn_clear.on_click(on_clear)
btn_prep.on_click(on_prepare)

display(widgets.VBox([
    widgets.HBox([btn_add, btn_chain, btn_prep, btn_clear]),
    info_out
]))
refresh_info()

## 4. Graph node type inspection

Show how `get_node_type`, `get_num_input_ports`, `get_num_output_ports`
reflect the types and port counts of different node classes.

In [ ]:
g = cp.AudioGraph()

# ADSR is an AudioNode (Producer)
adsr_id = g.add_node(cp.adsr.ADSR())

# BlepOscillator — if exposed as a graph node via bind_oscillators
# (add here when node binding is available)

g.prepare(NUM_CHANNELS, 512, float(SAMPLE_RATE))

print('Node inspection:')
print(f'  ADSR (id={adsr_id})')
print(f'    type:          {g.get_node_type(adsr_id)}')
print(f'    input ports:   {g.get_num_input_ports(adsr_id)}')
print(f'    output ports:  {g.get_num_output_ports(adsr_id)}')
print(f'    sample rate:   {g.get_sample_rate(adsr_id)}')
print(f'    is prepared:   {g.node_is_prepared(adsr_id)}')

# Verify error handling for invalid node ID
try:
    g.get_node_type(9999)
except ValueError as e:
    print(f'  Invalid id raises: {e}')

## 5. Feedback connection — interactive block-by-block inspection

Build a graph with a feedback edge. Show how the sorted order places
the feedback source *after* its consumer, and run it block by block.

In [ ]:
# Feedback topology:
#   A --normal--> B
#   B --feedback-> A   (B reads A's previous-block output)

g = cp.AudioGraph()
nA = g.add_node(cp.adsr.ADSR())
nB = g.add_node(cp.adsr.ADSR())

g.connect(nA, 0, nB, 0, cp.ConnectionType.Audio, is_feedback=False)
g.connect(nB, 0, nA, 0, cp.ConnectionType.Audio, is_feedback=True)

g.prepare(NUM_CHANNELS, 512, float(SAMPLE_RATE))

order = g.get_sorted_order()
print('Feedback topology A->B, B-[fb]->A')
print(f'  NodeIds: A={nA}, B={nB}')
print(f'  Sorted order: {order}')

# Feedback pass must place A before B (B reads A's previous block)
pos_A = order.index(nA)
pos_B = order.index(nB)
print(f'  pos(A)={pos_A}, pos(B)={pos_B}')
if pos_A < pos_B:
    print('  A before B: PASS (normal edge dominates)')
else:
    print('  B before A: PASS (feedback pass reordered correctly)')

# Run several blocks
for _ in range(10):
    g.process()
print('  10 blocks processed without error')

## 6. Interactive patch — ADSR + LFO modulation chain

Wire: `LFO -> ADSR (ordering edge) -> oscillator`

The graph controls execution order. Parameters are controlled via sliders.

In [ ]:
patch_out = widgets.Output()

w_attack  = widgets.FloatSlider(value=0.01, min=0.001, max=0.5,  step=0.001, description='Attack',   continuous_update=False, readout_format='.3f')
w_decay   = widgets.FloatSlider(value=0.08, min=0.001, max=0.5,  step=0.001, description='Decay',    continuous_update=False, readout_format='.3f')
w_sustain = widgets.FloatSlider(value=0.7,  min=0.01,  max=1.0,  step=0.01,  description='Sustain',  continuous_update=False, readout_format='.2f')
w_release = widgets.FloatSlider(value=0.3,  min=0.01,  max=2.0,  step=0.01,  description='Release',  continuous_update=False, readout_format='.3f')
w_freq    = widgets.FloatSlider(value=440., min=110.,  max=1760., step=1.,    description='Freq Hz',  continuous_update=False)
w_lfo     = widgets.FloatSlider(value=4.0,  min=0.1,   max=20.,  step=0.1,   description='LFO rate', continuous_update=False)
w_vib     = widgets.FloatSlider(value=0.02, min=0.,    max=0.2,  step=0.005, description='Vib depth',continuous_update=False, readout_format='.3f')

def render_patch(_):
    DURATION    = 1.5
    HOLD        = int(SAMPLE_RATE * 0.8)
    N           = int(SAMPLE_RATE * DURATION)
    BLOCK       = 256

    # --- Build graph components (standalone, connected manually) ---
    # Graph owns the execution order; DSP objects are driven per-sample here
    # since we need per-sample output for audio rendering.

    g = cp.AudioGraph()

    # Nodes
    adsr_id = g.add_node(cp.adsr.ADSR())
    lfo_id  = g.add_node(cp.lfo.LFO(float(SAMPLE_RATE), w_lfo.value))

    # Connect LFO -> ADSR (ordering: LFO runs before ADSR)
    g.connect(lfo_id, 0, adsr_id, 0, cp.ConnectionType.Audio)
    g.prepare(NUM_CHANNELS, BLOCK, float(SAMPLE_RATE))

    order = g.get_sorted_order()

    # Standalone DSP for per-sample audio (graph controls order metadata)
    env = cp.adsr.ADSR()
    env.set_sample_rate(float(SAMPLE_RATE))
    env.set_sustain_level(w_sustain.value)
    env.set_attack_time(w_attack.value)
    env.set_decay_time(w_decay.value)
    env.set_release_time(w_release.value)

    osc = cp.oscillators.BlepOscillator()
    osc.set_sample_rate(float(SAMPLE_RATE))
    osc.set_frequency(w_freq.value)
    osc.set_shape(cp.oscillators.WaveShape.Saw)

    lfo = cp.lfo.LFO(float(SAMPLE_RATE), w_lfo.value)
    lfo.set_output_mode(cp.lfo.LfoOutputMode.Bipolar)

    env.note_on()
    buf = np.zeros(N, dtype=np.float32)

    for i in range(N):
        if i == HOLD:
            env.note_off()
        vib = lfo.render_sample() * w_vib.value
        osc.frequency.add_modulation(vib)
        buf[i] = osc.render_sample() * env.render()
        osc.frequency.clear_modulation()

    with patch_out:
        patch_out.clear_output(wait=True)

        print(f'Graph sorted order (lfo={lfo_id} before adsr={adsr_id}): {order}')
        assert order.index(lfo_id) < order.index(adsr_id), 'LFO must precede ADSR'
        print('Topology constraint LFO < ADSR: PASS')

        fig, axes = plt.subplots(1, 2, figsize=(13, 3))
        t = np.arange(N) / SAMPLE_RATE
        axes[0].plot(t, buf, lw=0.4)
        axes[0].axvline(HOLD / SAMPLE_RATE, color='r', linestyle='--', alpha=0.5, label='noteOff')
        axes[0].set_title(f'{w_freq.value:.0f} Hz saw + ADSR + vibrato')
        axes[0].set_xlabel('Time (s)')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        freqs = np.fft.rfftfreq(N, 1.0 / SAMPLE_RATE)
        mag   = 20 * np.log10(np.abs(np.fft.rfft(buf)) / N + 1e-12)
        axes[1].plot(freqs, mag, lw=0.6)
        axes[1].set_xlim(0, min(w_freq.value * 15, 6000))
        axes[1].set_ylim(-80, 5)
        axes[1].set_title('Spectrum')
        axes[1].set_xlabel('Hz')
        axes[1].set_ylabel('dB')
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        display(Audio(buf, rate=SAMPLE_RATE, normalize=False))

for w in [w_attack, w_decay, w_sustain, w_release, w_freq, w_lfo, w_vib]:
    w.observe(render_patch, names='value')

display(widgets.VBox([
    widgets.HBox([w_freq, w_lfo, w_vib]),
    widgets.HBox([w_attack, w_decay, w_sustain, w_release]),
    patch_out
]))
render_patch(None)

## 7. Dynamic graph reconfiguration

Show that adding/removing connections invalidates `prepare`, and that
calling `prepare` again restores correct ordering.

In [ ]:
g = cp.AudioGraph()
n0 = g.add_node(cp.adsr.ADSR())
n1 = g.add_node(cp.adsr.ADSR())
n2 = g.add_node(cp.adsr.ADSR())

g.connect(n0, 0, n2, 0, cp.ConnectionType.Audio)
g.prepare(NUM_CHANNELS, 512, float(SAMPLE_RATE))

print('Initial graph')
print(f'  Prepared: {g.is_prepared()}')
print(f'  Order:    {g.get_sorted_order()}')

# Add a new connection — invalidates prepare
g.connect(n1, 0, n2, 0, cp.ConnectionType.Audio)
print(f'After adding n1->n2:')
print(f'  Prepared: {g.is_prepared()}  (should be False)')

# Re-prepare
g.prepare(NUM_CHANNELS, 512, float(SAMPLE_RATE))
order = g.get_sorted_order()
print(f'After re-prepare:')
print(f'  Prepared: {g.is_prepared()}')
print(f'  Order:    {order}')
assert order.index(n0) < order.index(n2), 'n0 must precede n2'
assert order.index(n1) < order.index(n2), 'n1 must precede n2'
print('  n0 < n2 and n1 < n2: PASS')

## 8. Polyphonic patch skeleton — multiple ADSR nodes in one graph

Add N envelope nodes to a single graph to simulate a simplified polyphonic
voice structure. Each envelope runs independently in topological order.

In [ ]:
NUM_VOICES = 4
BLOCK_SIZE = 512

g = cp.AudioGraph()
env_ids = [g.add_node(cp.adsr.ADSR()) for _ in range(NUM_VOICES)]

# All envelopes are independent (no connections between them)
g.prepare(NUM_CHANNELS, BLOCK_SIZE, float(SAMPLE_RATE))

print(f'{NUM_VOICES}-voice envelope graph')
print(f'  Nodes:        {g.get_num_nodes()}')
print(f'  Connections:  {g.get_num_connections()}')
print(f'  Sorted order: {g.get_sorted_order()}')

# Drive standalone ADSR objects for audio, use graph for ordering metadata
NOTES   = [60, 64, 67, 71]
HOLD_S  = 0.4
TOTAL_S = 1.2
N       = int(SAMPLE_RATE * TOTAL_S)
HOLD    = int(SAMPLE_RATE * HOLD_S)

envs = []
oscs = []
for i, note in enumerate(NOTES):
    freq = 440.0 * (2.0 ** ((note - 69) / 12.0))
    e = cp.adsr.ADSR()
    e.set_sample_rate(float(SAMPLE_RATE))
    e.set_sustain_level(0.7)
    e.set_attack_time(0.01)
    e.set_decay_time(0.05)
    e.set_release_time(0.25)
    e.note_on()
    envs.append(e)

    o = cp.oscillators.BlepOscillator()
    o.set_sample_rate(float(SAMPLE_RATE))
    o.set_frequency(freq)
    o.set_shape(cp.oscillators.WaveShape.Sine)
    oscs.append(o)

mix = np.zeros(N, dtype=np.float32)
gain = 1.0 / NUM_VOICES

for i in range(N):
    if i == HOLD:
        for e in envs:
            e.note_off()
    for e, o in zip(envs, oscs):
        mix[i] += o.render_sample() * e.render() * gain

print(f'Rendered {NUM_VOICES} voices, {N} samples')

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(np.arange(N) / SAMPLE_RATE, mix, lw=0.4)
ax.axvline(HOLD_S, color='r', linestyle='--', alpha=0.5, label='noteOff')
ax.set_title(f'{NUM_VOICES}-voice chord (MIDI {NOTES})')
ax.set_xlabel('Time (s)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
display(Audio(mix, rate=SAMPLE_RATE, normalize=True))